In [7]:
import pandas as pd
import glob, os
import json
from collections import defaultdict
import re

In [ ]:
input_pattern = "scimagojr *.csv"
## output_dir = "~/Downloads/scimago_filtered/"
# os.makedirs(os.path.expanduser(output_dir), exist_ok=True)
result = defaultdict(dict)

for filepath in sorted(glob.glob(os.path.expanduser(input_pattern))):
    year = os.path.basename(filepath).replace("scimagojr ", "").replace(".csv", "")
    df = pd.read_csv(filepath, sep=";", usecols=["Title", "Type", "Categories"])
    df = df[df["Type"] == "journal"]

    for _, row in df.iterrows():
        title = row["Title"]
        categories = row["Categories"]
        result[title][year] = categories

# Save to JSON
out_path = os.path.expanduser("scimago_by_journal.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

# for filepath in sorted(glob.glob(os.path.expanduser(input_pattern))):
#     year = os.path.basename(filepath).replace("scimagojr ", "").replace(".csv", "")
#     df = pd.read_csv(filepath, sep=";", usecols=["Title", "Type", "Categories"])
#     df = df[ df["Type"].str.lower() == "journal" ]
#     print( df )
#     #out_path = os.path.expanduser(f"{output_dir}scimago_{year}_filtered.csv")
#     #df.to_csv(out_path, index=False)
#     #print(f"Year {year}: {len(df)} journals → {out_path}")


In [7]:
in_path = os.path.expanduser("scimago_by_journal.json")
with open(in_path, "r", encoding="utf-8") as f:
    result = json.load(f)

def compress_journal(year_dict):
    compressed = {}
    prev_value = None
    for year in sorted(year_dict.keys()):
        value = year_dict[year]
        if value != prev_value:
            compressed[year] = value
            prev_value = value
    return compressed

compressed_result = {
    journal: compress_journal(years)
    for journal, years in result.items()
}

out_path = os.path.expanduser("scimago_by_journal_compressed.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(compressed_result, f, ensure_ascii=False, indent=2)

total_before = sum(len(v) for v in result.values())
total_after = sum(len(v) for v in compressed_result.values())
print(f"Entries before: {total_before}, after: {total_after} ({100*(1-total_after/total_before):.1f}% reduction)")

Entries before: 595106, after: 256509 (56.9% reduction)


In [ ]:
input_pattern = "CORE/CORE_*.csv"
result = {}  # maps title/acronym → ranking

def clean_title(title: str) -> str:
    prev = None
    while prev != title:
        prev = title
        title = re.sub(r'\s*\(.*?\)\s*$', '', title).strip()
    return title

for filepath in sorted(glob.glob(os.path.expanduser(input_pattern))):
    year = os.path.basename(filepath).replace("CORE_", "").replace(".csv", "")
    df = pd.read_csv(filepath, usecols=[1, 2, 4], header=0)
    df.columns = ["title", "acronym", "rank"]
    df = df.dropna(subset=["rank"])
    df["rank"] = df["rank"].str.strip()

    for _, row in df.iterrows():
        title   = clean_title(str(row["title"]).strip())
        acronym = str(row["acronym"]).strip()
        rank    = row["rank"]

        if title and title != "nan":
            if title not in result or year > list(result[title].keys())[-1]:
                result.setdefault(title, {})[year] = rank

        if acronym and acronym != "nan":
            if acronym not in result or year > list(result[acronym].keys())[-1]:
                result.setdefault(acronym, {})[year] = rank

out_path = os.path.expanduser("core_rankings.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"Done! {len(result)} keys written to {out_path}")

def compress_rankings(year_dict):
    compressed = {}
    prev_value = None
    for year in sorted(year_dict.keys()):
        value = year_dict[year]
        if value != prev_value:
            compressed[year] = value
            prev_value = value
    return compressed

compressed = {k: compress_rankings(v) for k, v in result.items()}

with open(os.path.expanduser("core_rankings_compressed.json"), "w", encoding="utf-8") as f:
    json.dump(compressed, f, ensure_ascii=False, indent=2)

processing #CORE/CORE_2008.csv
processing year #2008
processing #CORE/CORE_2013.csv
processing year #2013
processing #CORE/CORE_2014.csv
processing year #2014
processing #CORE/CORE_2017.csv
processing year #2017
processing #CORE/CORE_2018.csv
processing year #2018
processing #CORE/CORE_2020.csv
processing year #2020
processing #CORE/CORE_2021.csv
processing year #2021
processing #CORE/CORE_2023.csv
processing year #2023
Done! 3605 keys written to core_rankings.json


In [9]:
import gzip, json, base64, os

def compress_for_fflate(input_json_path: str, output_js_path: str, var_name: str = "DATA"):
    with open(os.path.expanduser(input_json_path), "r", encoding="utf-8") as f:
        raw = f.read()

    compressed = gzip.compress(raw.encode("utf-8"), compresslevel=9)
    b64 = base64.b64encode(compressed).decode("ascii")

    js = f'export const {var_name} = "{b64}";\n'

    with open(os.path.expanduser(output_js_path), "w", encoding="utf-8") as f:
        f.write(js)

    raw_kb      = len(raw.encode()) / 1024
    compressed_kb = len(compressed) / 1024
    b64_kb      = len(b64) / 1024
    print(f"Original:    {raw_kb:>10.1f} KB")
    print(f"Compressed:  {compressed_kb:>10.1f} KB  ({100*(1 - compressed_kb/raw_kb):.1f}% reduction)")
    print(f"Base64:      {b64_kb:>10.1f} KB  (~33% base64 overhead)")


# Usage
compress_for_fflate("scimago_by_journal_compressed.json", "scimago_binary.js",  var_name="SCIMAGO")
compress_for_fflate("core_rankings_compressed.json",      "core_binary.js",     var_name="CORE")

Original:       24061.4 KB
Compressed:      2293.9 KB  (90.5% reduction)
Base64:          3058.6 KB  (~33% base64 overhead)
Original:         240.0 KB
Compressed:        35.3 KB  (85.3% reduction)
Base64:            47.1 KB  (~33% base64 overhead)
